# PlaceMux Task 5 - Marketplace Integration & Company Portal v1

This notebook demonstrates the end-to-end matching workflow between Company Jobs and Student Applications, validating the baseline matcher, machine learning model, and ranking system.

In [1]:
import pandas as pd
import sys
import os
sys.path.append('../src')

from baseline_matcher import BaselineMatcher
from ranker import CandidateRanker

## 1. Load Real-Shaped Dataset

In [2]:
jobs_df = pd.read_csv('../data/jobs.csv')
students_df = pd.read_csv('../data/students.csv')

print(f"Loaded {len(jobs_df)} jobs and {len(students_df)} students.")
jobs_df.head(2)

Loaded 100 jobs and 500 students.


,job_id,title,required_skills,preferred_skills,minimum_skill_score,experience_required
0,J001,Machine Learning Engineer,"Data Engineering, MLOps",NaN,58,5
1,J002,Data Scientist,"GCP, SQL",NaN,55,1


## 2. Generate Match Vector and Threshold Validation
We use the `BaselineMatcher` to extract matching vectors, match scores, and a plain English reasoning.

In [3]:
matcher = BaselineMatcher(threshold=70.0)

sample_job = jobs_df.iloc[0]
sample_student = students_df.iloc[0]

print("Job Required Skills:", sample_job['required_skills'])
print("Student Verified Skills:", sample_student['verified_skills'])
print("-" * 40)

res = matcher.match(sample_job, sample_student)
print("Match Vector:", res['match_vector'])
print(f"Match Score: {res['match_score']:.2f}%")
print(f"Status: {res['status'].upper()}")
print("\nReasoning:\n", res['explanation'])

Job Required Skills: Data Engineering, MLOps
Student Verified Skills: Data Engineering, Java, AWS, React, SQL, NLP, Docker, Node.js
----------------------------------------
Match Vector: [1, 0]
Match Score: 50.00%
Status: REJECTED

Reasoning:
 Candidate rejected because:
✓ Data Engineering found
✗ MLOps missing
Overall Match Score = 50.00%
Average Verified Skill Score = 64.00


## 3. End-to-End Candidate Ranking
Using the `CandidateRanker` to process all applicants for a job, filter out rejected ones, and sort them.

In [4]:
ranker = CandidateRanker(matcher)
ranked_candidates = ranker.rank_candidates(sample_job, students_df)

print(f"Total Eligible Candidates: {len(ranked_candidates)}\n")
print("Top 3 Candidates:")
for i, candidate in enumerate(ranked_candidates[:3]):
    print(f"Rank {i+1} | Student {candidate['student_id']} | Score: {candidate['match_score']:.2f}%")

Total Eligible Candidates: 25

Top 3 Candidates:
Rank 1 | Student S316 | Score: 100.00%
Rank 2 | Student S154 | Score: 100.00%
Rank 3 | Student S121 | Score: 100.00%


## 4. Explainable ML Model (Logistic Regression)
Reviewing the logistic regression model trained on feature-engineered match data.

In [5]:
import json

with open('../models/metrics.json', 'r') as f:
    metrics = json.load(f)

print("Logistic Regression Evaluation Metrics:")
for k, v in metrics.items():
    print(f"{k}: {v*100:.2f}%")

Logistic Regression Evaluation Metrics:
Accuracy: 95.07%
Precision: 94.31%
Recall: 96.46%
F1 Score: 95.37%
False Positive Rate: 6.48%
False Negative Rate: 3.54%
